# PART 1 — Text-to-Math Agent Overview

### 1. What is a Text-to-Math Problem?
A text-to-math problem converts natural language questions into mathematical expressions and solves them.

### 2. Why Agents Are Useful?
Agents break problems into reasoning steps, choose tools like calculators, and reduce calculation errors.

### 3. Normal LLM vs Agent
LLM:
- Direct answer generation
- May hallucinate calculations

Agent:
- Think → Use Tool → Observe → Answer
- Accurate computation

In [2]:
############################################################
# PART 2 — TASK 2 — STEP 1: Imports & LLM Setup
############################################################

from langchain_community.llms import Ollama
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool

# Load lightweight Ollama model
llm = Ollama(
    model="llama3.2:1b",
    temperature=0
)

print("LLM Loaded ")

LLM Loaded 


In [ ]:

# PART 2 — TASK 2 — STEP 2: Calculator Tool

@tool
def calculator(expression: str) -> str:
    """
    Use this tool to solve mathematical expressions.
    Example input: 25*4+10
    """
    try:
        return str(eval(expression))
    except Exception as e:
        return str(e)

print("Calculator Tool Ready ")

Calculator Tool Ready 


In [4]:

# PART 2 — TASK 2 — STEP 3: Text to Math Tool


@tool
def text_to_math(problem: str) -> str:
    """
    Converts word problems into math expressions.
    """
    prompt = f"""
    Convert the following word problem into a mathematical expression only.

    Problem: {problem}
    Expression:
    """

    response = llm.invoke(prompt)
    return response.strip()

print("Text-to-Math Tool Ready ")

Text-to-Math Tool Ready 


In [5]:

# PART 2 — TASK 2 — STEP 4: Agent Creation


tools = [calculator, text_to_math]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3
)

print("Text-to-Math Agent Initialized ")

Text-to-Math Agent Initialized 


d:\Assignment_35_Text_to_Math\text_math_env\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [9]:
############################################################
# STEP: Clean LLM Math Expression
############################################################

import re

def clean_expression(expr: str) -> str:
    """
    Converts LLM output into valid Python math expression.
    """

    # lowercase
    expr = expr.lower()

    # replace math words/symbols
    expr = expr.replace("x", "*")
    expr = expr.replace("×", "*")

    # remove everything after '='
    if "=" in expr:
        expr = expr.split("=")[0]

    # remove non math characters
    expr = re.sub(r"[^0-9\.\+\-\*\/\(\)\% ]", "", expr)

    # remove spaces
    expr = expr.replace(" ", "")

    return expr

In [10]:
############################################################
# PART 2 — Stable Text-to-Math Solver
############################################################

def solve_text_math(problem: str):

    print("\nUser Problem:", problem)

    # STEP 1 — Generate expression
    prompt = f"""
    Convert the following word problem into ONLY a math expression.
    Do NOT include '=' or explanation.

    Problem: {problem}
    Expression:
    """

    raw_expression = llm.invoke(prompt).strip()
    print("LLM Output:", raw_expression)

    # STEP 2 — Clean expression
    expression = clean_expression(raw_expression)
    print("Clean Expression:", expression)

    # STEP 3 — Calculate
    result = calculator.run(expression)

    print("Final Answer:", result)
    return result

In [ ]:
# PART 2 — TASK 2 — STEP 5: Testing Agent
solve_text_math("If a student buys 5 pens costing 10 each, what is total cost?")
solve_text_math("What is 20 percent of 150?")
solve_text_math("What is 25 multiplied by 8 plus 10?")


User Problem: If a student buys 5 pens costing 10 each, what is total cost?
LLM Output: 5 x 10
Clean Expression: 5*10
Final Answer: 50

User Problem: What is 20 percent of 150?
LLM Output: 0.2 * 150
Clean Expression: 0.2*150
Final Answer: 30.0

User Problem: What is 25 multiplied by 8 plus 10?
LLM Output: 250 + 10
Clean Expression: 250+10
Final Answer: 260


'260'